In [1]:
!pip install opencv-python mtcnn --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.4 MB/s eta 0:00:00


In [43]:
from torchvision import models

In [21]:
import torch

In [2]:
import cv2
from mtcnn import MTCNN

In [3]:
import os

!unzip -q 'Images.zip' -d 'extracted_images'


In [4]:
path = 'extracted_images/Images/'

In [9]:
def crop_faces(image_path, output_prefix="face"):

    detector = MTCNN()
    image = cv2.imread(image_path)

    if image is not None:

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        detections = detector.detect_faces(image_rgb)

        x, y, width, height = detections[0]['box']
        x, y = max(0, x), max(0, y)
        cropped_face = image[y:y+height, x:x+width]
        return cropped_face


In [31]:
ls = []
for k in os.listdir(path):
    full_path = path + k
    rimg = crop_faces(full_path)
    rsz = cv2.resize(rimg, (224, 224))
    reshap_img = rsz.reshape(3, 224, 224)
    ls.append(reshap_img)

Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lz4/frame/__init__.py", line 753, in flush
    self._fp.flush()
ValueError: I/O operation on closed file.
Exception ignored in: <_io.BufferedReader>
Traceback (most recent call l

In [37]:
X_imgs = torch.FloatTensor(ls)

In [40]:
yls = []
for k in os.listdir(path):
    if k.startswith('tom'):
        yls.append(0)
    else:
        yls.append(1)


In [41]:
yls

[0, 1, 0, 0, 1, 0, 1, 0, 1, 1]

In [42]:
Y = torch.LongTensor(yls)

# Pretrained Model

In [45]:
model = models.vgg16(pretrained=True)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [49]:
model.classifier[6] = torch.nn.Linear(in_features=4096, out_features=2, bias=True)

In [64]:
for param in model.parameters():
    param.requires_grad = False

In [65]:
for p in model.classifier[6].parameters():
  p.requires_grad = True

# Training for new data

In [69]:
from torch.optim import Adam

In [70]:
opt = Adam(model.parameters(), lr=0.001)

In [71]:
lossfn = torch.nn.CrossEntropyLoss()

In [72]:
for epoch in range(30):
    opt.zero_grad()
    pred = model(X_imgs)
    loss = lossfn(pred,Y)
    loss.backward()
    opt.step()
    print(loss.item())

2.5569987297058105
3.5540976524353027
3.126413106918335
1.7711308002471924
3.02593731880188
3.3186748027801514
3.4600234031677246
1.1550390720367432
2.0483527183532715
1.6304610967636108
2.537306308746338
3.558859348297119
0.5567960143089294
1.6574831008911133
0.7056888341903687
1.7023214101791382
0.11644525825977325
1.142856240272522
0.004305919632315636
0.25642409920692444
1.3270950317382812
2.1355834007263184
0.20633795857429504
0.614199697971344
1.188814640045166
0.0017175026005133986
2.2911839485168457
0.6108092665672302
0.37986141443252563
1.96584153175354


In [73]:
torch.save(model.state_dict(), 'vgg_weights.pth')